# 💊 Pipeline Cào Dữ Liệu Nhà Thuốc Long Châu (Bronze Zone — Data Lakehouse)

Notebook này đóng gói toàn bộ quy trình cào và đóng gói dữ liệu thô từ **Nhà Thuốc Long Châu** (`nhathuoclongchau.com.vn`), đưa vào **Tầng Bronze (Raw Zone)** của hệ thống Data Lakehouse Dược Phẩm.

### 📋 Quy Chuẩn Tầng Bronze:
- **Nguồn dữ liệu (`source_name`)**: `Nhà Thuốc Long Châu`
- **Mức độ tin cậy (`trust_score`)**: `0.75`
- **Định dạng file**: Raw JSON nguyên bản từ `__NEXT_DATA__` SSR.
- **Metadata bắt buộc**: `source_url`, `ingestion_timestamp`, `file_hash` (SHA-256), `lineage_hash`.

## 1. Cấu Hình & Thiết Lập Môi Trường (Cần Chạy Đầu Tiên)

In [ ]:
import os
import sys
import time
import math
import json
import re
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
from tqdm.notebook import tqdm

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Thiết lập thư mục dự án
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'longchau_scraper':
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / 'bronze' / 'longchau'
QUEUE_FILE = PROJECT_ROOT / 'longchau_scraper' / 'url_queue_longchau.json'

# Tạo thư mục nếu chưa tồn tại
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
QUEUE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Cấu hình Scraping
BASE_URL = 'https://nhathuoclongchau.com.vn'
AZ_INDEX_URL = 'https://nhathuoclongchau.com.vn/thuoc/tra-cuu-thuoc-a-z'
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0'
]

REQUEST_DELAY_MIN = 0.5
REQUEST_DELAY_MAX = 1.2
MAX_RETRIES = 3
TIMEOUT_SECONDS = 15
SOURCE_NAME = 'Nhà Thuốc Long Châu'
TRUST_SCORE = 0.75

print('✅ Đã khởi tạo xong Cấu Hình Long Châu!')
print(f'📂 Thư mục chứa dữ liệu Bronze: {BRONZE_DIR}')
print(f'📋 File Quản lý Queue:           {QUEUE_FILE}')

## 2. Giai Đoạn 1: Khám Phá URL Thuốc (URL Discovery qua A-Z)

Tự động duyệt danh mục tra cứu A-Z (`/thuoc/tra-cuu-thuoc-a-z?alphabet={letter}&page={page}`), trích xuất danh sách sản phẩm và lưu vào file hàng đợi `url_queue_longchau.json` (**Resume-Safe**).

In [ ]:
def load_queue():
    if QUEUE_FILE.exists():
        try:
            with open(QUEUE_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f'⚠️ Warning: Lỗi đọc file queue ({e}), khởi tạo mới.')
    return {}

def save_queue(queue):
    with open(QUEUE_FILE, 'w', encoding='utf-8') as f:
        json.dump(queue, f, ensure_ascii=False, indent=2)

def get_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    }

def fetch_az_page(letter: str, page: int = 1):
    url = f'{AZ_INDEX_URL}?alphabet={letter}&page={page}'
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', resp.text, re.DOTALL)
                if match:
                    data = json.loads(match.group(1))
                    page_props = data.get('props', {}).get('pageProps', {})
                    return page_props.get('listDrugs', {})
        except Exception as e:
            pass
        time.sleep(attempt * 1.2)
    return None

def discover_urls(letters=None):
    if not letters:
        letters = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
    
    queue = load_queue()
    initial_count = len(queue)
    new_urls_count = 0

    print(f'🔍 Bắt đầu khám phá URL cho các chữ cái: {", ".join(letters)}')
    print(f'📋 Kích thước Queue hiện tại: {initial_count} items')

    for letter in letters:
        print(f'\n---> Quét chữ cái: "{letter}"')
        first_page_data = fetch_az_page(letter, page=1)
        if not first_page_data:
            print(f'❌ Không thể lấy trang 1 cho chữ cái "{letter}". Bỏ qua.')
            continue
        
        total_count = first_page_data.get('totalCount', 0)
        items = first_page_data.get('items', [])
        total_pages = math.ceil(total_count / 20) if total_count > 0 else (1 if items else 0)
        print(f'  📊 Tổng số thuốc tìm thấy: {total_count} ({total_pages} trang)')

        for item in items:
            slug = item.get('slug')
            if slug:
                full_url = f'{BASE_URL}/{slug.lstrip("/")}' if not slug.startswith('http') else slug
                if full_url not in queue:
                    queue[full_url] = {
                        'status': 'pending',
                        'discovered_at': datetime.now(timezone.utc).isoformat(),
                        'name': item.get('webName') or item.get('name') or '',
                        'attempts': 0,
                        'error': None
                    }
                    new_urls_count += 1

        if total_pages > 1:
            pbar = tqdm(range(2, total_pages + 1), desc=f'  Trang cho "{letter}"', unit='page')
            for page in pbar:
                time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))
                page_data = fetch_az_page(letter, page=page)
                if page_data:
                    for item in page_data.get('items', []):
                        slug = item.get('slug')
                        if slug:
                            full_url = f'{BASE_URL}/{slug.lstrip("/")}' if not slug.startswith('http') else slug
                            if full_url not in queue:
                                queue[full_url] = {
                                    'status': 'pending',
                                    'discovered_at': datetime.now(timezone.utc).isoformat(),
                                    'name': item.get('webName') or item.get('name') or '',
                                    'attempts': 0,
                                    'error': None
                                }
                                new_urls_count += 1
                if page % 5 == 0:
                    save_queue(queue)
        save_queue(queue)

    print('\n==========================================')
    print('✅ Hoàn thành khám phá URL!')
    print(f'➕ URL mới thêm vào: {new_urls_count}')
    print(f'📋 Tổng số URL trong Queue: {len(queue)}')
    print(f'💾 Queue lưu tại: {QUEUE_FILE}')

### 🚀 Thực Thi Khám Phá URL

- `discover_urls(letters=None)`: Quét tự động tất cả 26 chữ cái từ A đến Z.

In [ ]:
# Thực thi khám phá URL cho TOÀN BỘ kho thuốc (A đến Z)
discover_urls(letters=None)

## 3. Giai Đoạn 2: Cào Chi Tiết & Đóng Gói Dữ Liệu Tầng Bronze (Bronze Ingestion)

Hàm `scrape_bronze()` sẽ đọc danh sách URL `pending` từ queue, tải đối tượng `pageProps`, đính kèm 5 trường Metadata tầng Bronze và lưu file raw JSON vào `bronze/longchau/`.

In [ ]:
def clean_slug_filename(url: str, sku: str = '') -> str:
    slug = url.split('/')[-1].replace('.html', '')
    slug_clean = re.sub(r'[^a-zA-Z0-9_-]', '_', slug)
    url_hash = hashlib.sha256(url.encode('utf-8')).hexdigest()[:8]
    if sku:
        return f'{sku}_{slug_clean}_{url_hash}.json'
    return f'{slug_clean}_{url_hash}.json'

def fetch_product_data(url: str):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', resp.text, re.DOTALL)
                if match:
                    raw_json_str = match.group(1)
                    data = json.loads(raw_json_str)
                    page_props = data.get('props', {}).get('pageProps', {})
                    return page_props, raw_json_str
                else:
                    return None, 'Không tìm thấy thẻ __NEXT_DATA__'
            elif resp.status_code == 404:
                return None, 'HTTP 404 Not Found'
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, str(e)
        time.sleep(attempt * 1.5)
    return None, 'Quá số lần thử lại'

def scrape_bronze(limit: int = None, retry_failed: bool = False):
    queue = load_queue()
    target_urls = [url for url, info in queue.items() if info.get('status') == 'pending' or (retry_failed and info.get('status') == 'failed')]

    if not target_urls:
        print('✨ Không có URL nào ở trạng thái pending trong Queue!')
        return

    if limit:
        target_urls = target_urls[:limit]

    print(f'🚀 Bắt đầu cào dữ liệu Bronze cho {len(target_urls)} sản phẩm (Limit: {limit or "Tất cả"})')
    print(f'📂 Lưu file tại: {BRONZE_DIR}')

    success_count = 0
    fail_count = 0
    pbar = tqdm(target_urls, desc='Scraping Bronze Zone', unit='doc')

    for i, url in enumerate(pbar):
        queue[url]['attempts'] = queue[url].get('attempts', 0) + 1
        page_props, raw_or_err = fetch_product_data(url)

        if page_props:
            product = page_props.get('product') or {}
            sku = product.get('sku', '')
            raw_data_str = json.dumps(page_props, ensure_ascii=False)
            file_hash = hashlib.sha256(raw_data_str.encode('utf-8')).hexdigest()
            ingestion_time = datetime.now(timezone.utc).isoformat()
            lineage_hash = hashlib.sha256(f'{url}_{file_hash}_{ingestion_time}'.encode('utf-8')).hexdigest()

            bronze_record = {
                'source_name': SOURCE_NAME,
                'trust_score': TRUST_SCORE,
                'source_url': url,
                'ingestion_timestamp': ingestion_time,
                'file_hash': file_hash,
                'lineage_hash': lineage_hash,
                'raw_data': page_props
            }

            filename = clean_slug_filename(url, sku)
            file_path = BRONZE_DIR / filename
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(bronze_record, f, ensure_ascii=False, indent=2)

            queue[url]['status'] = 'done'
            queue[url]['scraped_at'] = ingestion_time
            queue[url]['file_path'] = str(file_path)
            queue[url]['error'] = None
            success_count += 1
        else:
            queue[url]['status'] = 'failed'
            queue[url]['error'] = str(raw_or_err)
            fail_count += 1

        if (i + 1) % 5 == 0:
            save_queue(queue)
        time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))

    save_queue(queue)
    print('\n==========================================')
    print('🎉 Hoàn thành phiên cào Bronze!')
    print(f'✅ Thành công: {success_count}')
    print(f'❌ Thất bại: {fail_count}')
    print(f'💾 Tổng số file tại Bronze: {len(list(BRONZE_DIR.glob("*.json")))}')


### 🚀 Thực Thi Cào Dữ Liệu Bronze (Cào Toàn Bộ Sản Phẩm)

- `scrape_bronze(limit=None)`: Cào tất cả các sản phẩm đang có trong hàng đợi Queue.

In [ ]:
# Thực thi cào dữ liệu Bronze cho TOÀN BỘ sản phẩm trong queue
scrape_bronze(limit=None)

## 4. Kiểm Tra & Trích Xuất Mẫu Dữ Liệu Bronze (Data Verification)

Đọc kiểm tra 1 file JSON ngẫu nhiên trong `bronze/longchau/` để xác minh 5 trường metadata bắt buộc.

In [ ]:
bronze_files = list(BRONZE_DIR.glob('*.json'))
if bronze_files:
    sample_file = bronze_files[0]
    print(f'📄 Đọc kiểm tra file mẫu: {sample_file.name}\n')
    with open(sample_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print('--- 1. METADATA TẦNG BRONZE ---')
    print(f'Source Name : {data.get("source_name")}')
    print(f'Trust Score : {data.get("trust_score")}')
    print(f'Source URL  : {data.get("source_url")}')
    print(f'Timestamp   : {data.get("ingestion_timestamp")}')
    print(f'File Hash   : {data.get("file_hash")}')
    
    product = data.get('raw_data', {}).get('product', {})
    print('\n--- 2. THÔNG TIN SẢN PHẨM THÔ ---')
    print(f'SKU         : {product.get("sku")}')
    print(f'Tên thuốc   : {product.get("webName")}')
    print(f'Số đăng ký  : {product.get("registNum")}')
    print(f'Dạng bào chế: {product.get("dosageForm")}')
    print(f'Thành phần  : {product.get("ingredient")}')
    print(f'Cảnh báo    : {product.get("warning")}')
else:
    print('⚠️ Chưa có file nào trong thư mục bronze/longchau/')